In [ ]:
source("Main.R")
source("Conf.R")


In [ ]:
res <- read.csv(textfilesPath("ME_LMBetaFDRALL.csv.gz"), row.names = 1)

In [ ]:
head(res)

In [ ]:
rownames(res) = sapply(rownames(res), function(x){strsplit(x,"_")[[1]][2]})

In [ ]:
KOGenes = rownames(res)
testedGenes = colnames(res)

In [ ]:
length(testedGenes)

In [ ]:
length(unique(KOGenes))

In [ ]:
length(KOGenes[which(KOGenes %in% testedGenes)])

In [ ]:
GeneMod <- read.csv(referencePath("ME_GeneModules_leiden_11_Modules.csv"), row.names = 1)

In [ ]:
GuideMod <- read.csv(referencePath("ME_GuideModules_leiden_6_Modules.csv"), row.names = 1)

In [ ]:
head(GuideMod)

In [ ]:
head(GeneMod)

In [ ]:
selGuides <- GuideMod$GuideName
selGenes <- GeneMod$GeneName

In [ ]:
length(selGenes)

In [ ]:
length(selGuides)

In [ ]:
length(which(selGuides %in% selGenes))

In [ ]:
k = selGuides[which(selGuides %in% selGenes)]

In [ ]:
ss = GeneMod[k,c("GeneName","NewGeneGroup")]
colnames(ss) = c("Gene", "NewGeneGroup")
ss$NewGeneGroup = factor(ss$NewGeneGroup, levels = paste0("GP_",1:11))

In [ ]:
hh = GuideMod[k,c("GuideName","NewGuideGroup")]
colnames(hh) = c("Gene", "NewGuideGroup")

In [ ]:
hh = merge(hh, ss, by="Gene")

In [ ]:
hh[order(hh$NewGuideGroup, hh$NewGeneGroup),]

In [ ]:
resSel = res[selGuides,]

In [ ]:
nKOsSign = sapply(resSel, function(x){length(x[x<0.1])})

In [ ]:
myDF = data.frame("Gene"=names(nKOsSign),
           "NumSgnKO" = nKOsSign)
myDF$isE3="Not E3"
myDF[myDF$Gene %in% KOGenes[which(KOGenes %in% testedGenes)],  "isE3"] = "E3"

In [ ]:
table(myDF$isE3)

In [ ]:
E3Counts = myDF[myDF$isE3 == "E3","NumSgnKO"]
NotE3Counts = myDF[myDF$isE3 == "Not E3","NumSgnKO"]
cdfE3 = ecdf(E3Counts)
cdfNotE3 = ecdf(NotE3Counts)

In [ ]:
length(E3Counts)

In [ ]:
length(NotE3Counts)

In [ ]:
ks.test(E3Counts, NotE3Counts, alternative="greater")

In [ ]:
minMax <- seq(min(E3Counts, NotE3Counts),
              max(E3Counts, NotE3Counts), length.out=length(NotE3Counts)) 
x0 <- minMax[which( abs(cdfE3(minMax) - cdfNotE3(minMax)) == max(abs(cdfE3(minMax) - cdfNotE3(minMax))) )] 
y0 <- cdfE3(x0) 
y1 <- cdfNotE3(x0) 


In [ ]:
ggplot(myDF, aes(x = NumSgnKO, group = isE3, colour = isE3, linetype=isE3))+
  stat_ecdf(size=1) +
  xlab("Number of KOs (within knockout modules) that affected the gene") +
  ylab("Cumulitive Distibution") +
  geom_segment(aes(x = x0[1], y = y0[1], xend = x0[1], yend = y1[1]),
               linetype = "dashed", color = "red") +
  geom_point(aes(x = x0[1] , y= y0[1]), color="red", size=1) +
  geom_point(aes(x = x0[1] , y= y1[1]), color="red", size=1) +
  ggtitle("K-S Test: E3 / Not E3")+theme_minimal()

In [ ]:
options(repr.plot.width=10, repr.plot.height=5)
ggplot(data=myDF, aes(x=NumSgnKO, color=isE3, fill=isE3)) +
    geom_density(alpha=0.5)+
    theme_bw()+theme(axis.text.x = element_text(angle = 90, hjust = 1, size=12),
                     axis.text.y = element_text(size=18),
                     axis.title = element_text(size=20) )+
    scale_fill_manual(values = c("blue", "red"))+
    scale_color_manual(values = c("blue", "red"))+
    xlab("Number of KOs that affected the gene")+ylab("Density")

In [ ]:
library("ggpubr")
ggplot(myDF, aes(x=isE3, y=NumSgnKO)) + 
     geom_violin(trim=FALSE, aes(fill=isE3))+
     geom_boxplot()+theme_bw()+ 
     #stat_summary(fun.data = mean, geom = "text", size=6)+
        stat_compare_means(method="t.test",paired = FALSE,
                                comparisons = list(c("E3", "Not E3")),
                                color="red",
                                label.y = c(30), aes(label = ..p.signif..),method.args = list(alternative = "less"))+
     scale_fill_brewer(palette = "Pastel1")+
ylab("Number of KOs that affected the gene")+
theme(axis.text = element_text(size=15),
              axis.title =  element_text(size=16))+ylim(0,50) +xlab("") 

In [ ]:
sample1<-rnorm(1000)
sample2<-rnorm(1000, 2, 2)

group <- c(rep("sample1", length(sample1)), rep("sample2", length(sample2)))
dat <- data.frame(KSD = c(sample1,sample2), group = group)
cdf1 <- ecdf(sample1) 
cdf2 <- ecdf(sample2) 



In [ ]:
head(dat)

In [ ]:
minMax <- seq(min(sample1, sample2), max(sample1, sample2), length.out=length(sample1)) 
x0 <- minMax[which( abs(cdf1(minMax) - cdf2(minMax)) == max(abs(cdf1(minMax) - cdf2(minMax))) )] 
y0 <- cdf1(x0) 
y1 <- cdf2(x0) 


ggplot(dat, aes(x = KSD, group = group, colour = group, linetype=group))+
  stat_ecdf(size=1) +
  xlab("mm") +
  ylab("Cumulitive Distibution") +
  geom_segment(aes(x = x0[1], y = y0[1], xend = x0[1], yend = y1[1]),
               linetype = "dashed", color = "red") +
  geom_point(aes(x = x0[1] , y= y0[1]), color="red", size=1) +
  geom_point(aes(x = x0[1] , y= y1[1]), color="red", size=1) +
  ggtitle("K-S Test: Sample 1 / Sample 2")